# 16 — Estudo Integrador

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 16/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/16_estudo_integrador.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_estudo_integrador.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Integrar o conteúdo sem alterar projetos existentes.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
ETL + agregação + validação + modelo baseline, com caminhos locais e governança.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
from pathlib import Path
import pandas as pd
fonte = Path('datasets/flights.csv')
df = pd.read_csv(fonte)
exigidos = ['carrier','distance','flight']
assert set(exigidos).issubset(df.columns)
limpo = df.copy()
limpo['distance'] = pd.to_numeric(limpo['distance'],errors='coerce')
limpo = limpo.dropna(subset=['carrier','distance'])
limpo = limpo[limpo['distance'] > 0]
sumario = (limpo.groupby('carrier',as_index=False)
           .agg(voos=('carrier','size'),distancia_media=('distance','mean'))
           .sort_values(['voos','carrier'],ascending=[False,True],kind='stable'))
print(sumario.head(10).round(2).to_string(index=False))


carrier  voos  distancia_media
     AS  3784          1218.06
     WN  1394           909.66
     OO  1186           600.76
     DL  1082          1566.14
     UA  1051          1454.62
     AA   482          1739.23
     US   367          1525.51
     B6   214          1635.31
     VX   186           759.01
     F9   181          1033.72


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
assert not sumario.empty
assert int(sumario['voos'].sum()) == len(limpo)
assert sumario['distancia_media'].notna().all()
print('Total após ETL:',len(limpo),'| rejeitados:',len(df)-len(limpo))
print('Baseline: contagem e média descritiva, sem avaliação de modelo preditivo.')


Total após ETL: 10000 | rejeitados: 0
Baseline: contagem e média descritiva, sem avaliação de modelo preditivo.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 16 reuniu conceitos e verificações de **ETL, qualidade, agregação e baseline**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** linhagem e validação. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **linhagem e validação** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **linhagem e validação** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Laboratório opcional — Spark local real

Requer `RUN_SPARK_LABS=1` e `SPARK_LAB_PYTHON` apontando para um ambiente isolado com PySpark e Java. O teste falha explicitamente se executado incorretamente; resultados locais não comprovam cluster distribuído.


In [4]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import StructType,StructField,StringType,DoubleType
import pandas as pd
s=SparkSession.builder.master('local[2]').appName('ep-area10-16').config('spark.ui.enabled','false').getOrCreate()
try:
    s.sparkContext.setLogLevel('ERROR')
    voos=pd.read_csv('datasets/flights.csv').dropna(subset=['carrier','distance']).head(300)
    pares=[(str(x.carrier),float(x.distance)) for x in voos.itertuples()]
    df=s.createDataFrame(pares,StructType([StructField('carrier',StringType()),StructField('distance',DoubleType())]))
    real={r['carrier']:r['n'] for r in df.groupBy('carrier').agg(F.count('*').alias('n')).collect()}
    local=voos.groupby('carrier').size().to_dict()
    assert real==local,(real,local)
    print('Spark agrupamento integrado, contagem por companhia:',real)
    print('SPARK_LAB_OK_16')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 16 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_16' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Spark agrupamento integrado, contagem por companhia: {'UA': 36, 'AA': 5, 'B6': 7, 'DL': 29, 'OO': 39, 'F9': 6, 'US': 9, 'HA': 2, 'AS': 122, 'VX': 7, 'WN': 38}
SPARK_LAB_OK_16


## Exportação de resultados

Resumo agregado e reproduzível; os registros originais permanecem intactos.


In [5]:
from pathlib import Path
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
sumario.to_csv(OUT/'16_resumo_distancia_por_companhia.csv',index=False)
print('Exportado: outputs/16_resumo_distancia_por_companhia.csv')


Exportado: outputs/16_resumo_distancia_por_companhia.csv
